<a href="https://colab.research.google.com/github/IKKNIGHT/Protein_interactions/blob/main/Phase5_RFdiffusion_Sweep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 5: run RFdiffusion on the design specs
Reads `design_specs/*.json` (from the Phase 3/4 notebook) and writes backbones to `Protein_interactions/rfdiff_outputs/<spec name>/`.
**Use an A100 runtime.**

Cell 2 installs RFdiffusion the **same way the working MYC notebook does** (`RFDiffusion_gnn.ipynb` in the MYC-PROTAC repo): straight into the Colab kernel python with pip, a DGL wheel cascade, and the repo's own `download_models.sh` for weights (3 x 484 MB, cached to Drive). No conda. **Cell 2 is CPU-only and that is correct** — the GPU is used only in Cell 3/4. First run installs + downloads (a few min); reruns skip both. If DGL can't install it falls back to SE3Transformer (slower but works).

In [ ]:
# ── Cell 1: Mount, settings, pick specs ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, shutil, subprocess, time

BASE = '/content/drive/MyDrive/Protein_interactions'
SPEC_DIR, RFD_DRIVE = f'{BASE}/design_specs', f'{BASE}/rfdiff_outputs'
os.makedirs(RFD_DRIVE, exist_ok=True)

TARGET_FILTER = None   # None = ALL targets in design_specs/ (sweep); or a list of names                                   # Gate 1 = MDM2/p53 only
ARMS          = ['gnn', 'oracle', 'baseline', 'scrambled']
NUM_DESIGNS   = 15     # sweep depth (gate used 30)                                         # backbones per arm (~40 s each on an A100)

specs = [json.load(open(p)) for p in sorted(glob.glob(f'{SPEC_DIR}/*.json'))]
specs = [s for s in specs if (TARGET_FILTER is None or s['target'] in TARGET_FILTER) and s['arm'] in ARMS]
# ordering: single-pocket / peptide-groove targets first, multi-surface last
ORDER = ['MDM2','VHL','KEAP1','BCL2','BCL2L1 (Bcl-xL)','XIAP (BIR3)','MEN1','CREBBP (KIX)','RAD51','HRAS','ARF1','VCL','PCNA','GNAI1','CTNNB1','CDK2','BRCA1','FAS','JUN','MYC']
rank = {t: i for i, t in enumerate(ORDER)}
specs = sorted(specs, key=lambda s: (rank.get(s['target'], 99), s['arm']))
print(f'{len(specs)} specs selected:')
for s in specs: print(f"  {s['name']:<22} hotspots {s['hotspots']}  contig {s['contig']}")
assert specs, 'no specs found - run the Phase 3/4 notebook first'
subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', shell=True)

Mounted at /content/drive
80 specs selected:
  5_1YCR_baseline        hotspots ['A109', 'A27', 'A104', 'A26', 'A47']  contig A25-109/0 40-60
  5_1YCR_gnn             hotspots ['A100', 'A99', 'A96', 'A54', 'A97']  contig A25-109/0 40-60
  5_1YCR_oracle          hotspots ['A96', 'A93', 'A94', 'A99', 'A73']  contig A25-109/0 40-60
  5_1YCR_scrambled       hotspots ['A46', 'A45', 'A44', 'A39', 'A56']  contig A25-109/0 40-60
  33_1LM8_baseline       hotspots ['V137', 'V91', 'V62', 'V66', 'V135']  contig V60-209/0 40-60
  33_1LM8_gnn            hotspots ['V162', 'V165', 'V166', 'V169', 'V163']  contig V60-209/0 40-60
  33_1LM8_oracle         hotspots ['V110', 'V108', 'V112', 'V109', 'V73']  contig V60-209/0 40-60
  33_1LM8_scrambled      hotspots ['V201', 'V195', 'V196', 'V197', 'V191']  contig V60-209/0 40-60
  34_2FLU_baseline       hotspots ['X420', 'X467', 'X418']  contig X326-332/0 X362-379/0 X390-395/0 X406-431/0 X437-477/0 X487-500/0 X508-522/0 X536-544/0 X556-569/0 X603-609/0 40-60
 

CompletedProcess(args='nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', returncode=0)

In [ ]:
# ── Cell 2: RFdiffusion setup via conda py3.10 + REAL DGL (self-contained, idempotent) ──
# WHY CONDA: RFdiffusion calls dgl.graph() during EVERY design step (Track_module.py -> make_full_graph),
# so real DGL is required - a stub/shim fails at inference. DGL has no wheel for Colab's py3.13/torch2.11,
# so we build a py3.10 env (torch 2.1 / cuda 12.1 / dgl from dglteam). This is the setup that produced
# the working MDM2 gate run. Re-running this cell is cheap: it skips the build once the env imports.
import os, sys, time, subprocess, shutil
_t0 = time.time()
RFD, RFD_SCRIPT = '/content/RFdiffusion', '/content/RFdiffusion/scripts/run_inference.py'
WEIGHTS_DIR = f'{RFD}/models'; WEIGHTS = f'{WEIGHTS_DIR}/Complex_base_ckpt.pt'
WEIGHTS_DRIVE = f'{BASE}/rfdiff_weights'
CONDA = '/opt/miniforge'; ENVPY = f'{CONDA}/envs/rfd/bin/python'
os.makedirs(WEIGHTS_DIR, exist_ok=True); os.makedirs(WEIGHTS_DRIVE, exist_ok=True)

def sh(cmd, tail=2500, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    return r.returncode, (r.stdout + r.stderr)[-tail:]

def run(cmd, label=''):
    print(f'  {label or cmd[:55]}...', end=' ', flush=True)
    rc, out = sh(cmd); print('OK' if rc == 0 else 'FAILED')
    if rc: print(out[-1200:])
    return rc

import torch
print(f'kernel PyTorch {torch.__version__} | CUDA {torch.version.cuda} | GPU {torch.cuda.is_available()}')
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> A100, then re-run.'

# ---- 0. remove any leftover dgl SHIM from the kernel python (it breaks nothing, but avoid confusion) ----
_shim = '/usr/local/lib/python3.13/dist-packages/dgl'
if os.path.isdir(_shim) and os.path.exists(f'{_shim}/__init__.py') and 'shim' in open(f'{_shim}/__init__.py').read():
    shutil.rmtree(_shim); print('  removed stale dgl shim from kernel python')

# ---- 1. clone RFdiffusion (re-clone if incomplete) ----
if os.path.exists(RFD) and not os.path.exists(f'{RFD}/setup.py'):
    print('  incomplete clone - re-cloning'); shutil.rmtree(RFD)
if not os.path.exists(RFD):
    run(f'git clone --depth 1 https://github.com/RosettaCommons/RFdiffusion.git {RFD}', 'cloning RFdiffusion')

ENV = dict(os.environ, PYTHONPATH=RFD, DGLBACKEND='pytorch')
def rfd_ok():
    if not os.path.exists(ENVPY): return False, 'env not built'
    rc, out = sh(f'{ENVPY} -c "import os; os.environ[\'DGLBACKEND\']=\'pytorch\'; '
                 f'import torch, dgl, hydra, e3nn, se3_transformer; from rfdiffusion.inference import utils; '
                 f'assert torch.cuda.is_available(); print(chr(82)+chr(70)+chr(68)+chr(79)+chr(75))"',
                 cwd=RFD, env=dict(ENV, PATH=f'{CONDA}/envs/rfd/bin:' + os.environ['PATH']))
    return ('RFDOK' in out), out

# ---- 2. build the py3.10 env with REAL dgl (skip if it already imports) ----
ok, _ = rfd_ok()
if ok:
    print(f'  rfd env already works - skipping build  ({time.time()-_t0:.0f}s)')
else:
    if not os.path.exists(f'{CONDA}/bin/mamba'):
        print('  installing Miniforge (brings mamba)...')
        sh('wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh '
           '-O /tmp/mf.sh && bash /tmp/mf.sh -b -p /opt/miniforge')
    MB = f'{CONDA}/bin/mamba'
    print('  creating py3.10 env (torch 2.1 / cuda 12.1 / dgl) - a few minutes...', flush=True)
    _ts = time.time()
    rc, out = sh(f'{MB} create -y -n rfd -c pytorch -c nvidia -c dglteam -c conda-forge '
                 f'python=3.10 pytorch=2.1.* pytorch-cuda=12.1 dgl', tail=3000)
    print(f'    env solve rc {rc} in {(time.time()-_ts)/60:.1f} min')
    if rc: print(out[-1800:])
    pip = f'{ENVPY} -m pip install -q'
    run(f'{pip} hydra-core omegaconf pyrsistent e3nn opt_einsum decorator icecream biopython scipy', 'python deps')
    run(f'{pip} --no-cache-dir --no-deps {RFD}/env/SE3Transformer', 'SE3Transformer (from repo)')
    run(f'{pip} --no-deps -e {RFD}', 'RFdiffusion (editable)')
    ok, out = rfd_ok(); print('  import check:', out.strip()[-250:])

# ---- 3. weights: restore from Drive, else aria2c from the JHU mirror; require the full ~484 MB ----
subprocess.run('apt-get -qq install -y aria2', shell=True, capture_output=True)
H = {'Complex_base_ckpt.pt': 'e29311f6f1bf1af907f9ef9f44b8328b'}
good = lambda p: os.path.exists(p) and os.path.getsize(p) > 450e6
for f, h in H.items():
    loc, drv = f'{WEIGHTS_DIR}/{f}', f'{WEIGHTS_DRIVE}/{f}'
    for _ in range(3):
        if good(loc): break
        if good(drv): subprocess.run(f'cp "{drv}" "{loc}"', shell=True); time.sleep(2)
        else: break
    if not good(loc):
        if os.path.exists(loc): os.remove(loc)
        print(f'  downloading {f}...', flush=True)
        subprocess.run(f'aria2c -x16 -s16 -k1M --allow-overwrite=true -d {WEIGHTS_DIR} -o {f} '
                       f'https://ipd.graylab.jhu.edu/RFdiffusion/{h}/{f}', shell=True)
        if good(loc) and not good(drv): subprocess.run(f'cp "{loc}" "{drv}"', shell=True)
    print(f'  {f}: {os.path.getsize(loc)/1e6:.0f} MB' if os.path.exists(loc) else f'  {f}: MISSING')

# ---- 4. verify and export the vars the run cells use ----
ok, out = rfd_ok()
print(f'\nRFdiffusion importable (real DGL): {ok}  |  weights ok: {good(WEIGHTS)}')
print(f'Cell 2 total: {(time.time()-_t0)/60:.1f} min')
assert ok and good(WEIGHTS), 'setup incomplete - paste the import check output above'
PYBIN = ENVPY
ENV = dict(os.environ, PYTHONPATH=RFD, DGLBACKEND='pytorch',
           PATH=f'{CONDA}/envs/rfd/bin:' + os.environ['PATH'])
print('Using python:', PYBIN)

kernel PyTorch 2.11.0+cu130 | CUDA 13.0 | GPU True
  incomplete clone - re-cloning
  cloning RFdiffusion... OK
  installing Miniforge (brings mamba)...
  creating py3.10 env (torch 2.1 / cuda 12.1 / dgl) - a few minutes...
    env solve rc 0 in 1.8 min
  python deps... OK
  SE3Transformer (from repo)... OK
  RFdiffusion (editable)... OK
  import check: torch.export`.
  warnings.warn(
/content/RFdiffusion/rfdiffusion/Track_module.py:236: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
  downloading Complex_base_ckpt.pt...
  Complex_base_ckpt.pt: 484 MB

RFdiffusion importable (real DGL): True  |  weights ok: True
Cell 2 total: 5.0 min
Using python: /opt/miniforge/envs/rfd/bin/python


In [ ]:
# ── Cell 3: Smoke test - ONE design from the first spec ───────────────────────
import sys; sys.path.insert(0, BASE)  # rfdiffusion_shared.py lives at the repo root
from rfdiffusion_shared import rfd_cmd, run_stream
ENV = dict(os.environ, PYTHONPATH=RFD, DGLBACKEND='pytorch')

os.makedirs('/content/rfd_smoke', exist_ok=True)
t0 = time.time(); rc, tail = run_stream(
    rfd_cmd(PYBIN, RFD_SCRIPT, specs[0], '/content/rfd_smoke/test', 1, WEIGHTS),
    '/content/rfd_smoke/log.txt', RFD, ENV)
pdbs = glob.glob('/content/rfd_smoke/test*.pdb')
print(f'smoke test rc={rc}, {len(pdbs)} pdb, {time.time()-t0:.0f}s')
if pdbs:
    from collections import OrderedDict
    ch = OrderedDict()
    for l in open(pdbs[0]):
        if l.startswith('ATOM') and l[12:16].strip() == 'CA': ch.setdefault(l[21], []).append(int(l[22:26]))
    print('chain layout (chain: n residues, first-last):', {c: (len(v), v[0], v[-1]) for c, v in ch.items()})
    print('expected: target residues keep original numbering, binder chain is LAST and numbered from 1')
else:
    print(tail)
assert pdbs, 'smoke test failed - paste the log tail above'

    [2026-10-16 03:05:02,569][__main__][INFO] - Making design /content/rfd_smoke/test_0
    [2026-10-16 03:05:48,551][__main__][INFO] - Finished design in 0.77 minutes
smoke test rc=0, 1 pdb, 74s
chain layout (chain: n residues, first-last): {'B': (41, 1, 41), 'A': (85, 25, 109)}
expected: target residues keep original numbering, binder chain is LAST and numbered from 1


In [4]:
# ── Cell 4: Run every selected spec (resumable; outputs copied to Drive after each spec) ──
summary = []
for s in specs:
    name = s['name']; loc = f'/content/rfdiff_outputs/{name}'; drv = f'{RFD_DRIVE}/{name}'; os.makedirs(loc, exist_ok=True); os.makedirs(drv, exist_ok=True)
    for f in glob.glob(f'{drv}/*'):                                   # restore anything already on Drive
        if not os.path.exists(f'{loc}/{os.path.basename(f)}'): shutil.copy2(f, loc)
    have = len(glob.glob(f'{loc}/design_*.pdb'))
    if have < NUM_DESIGNS:
        print(f'\n{name}: {have} done, running {NUM_DESIGNS - have} more'); t0 = time.time()
        rc, tail = run_stream(rfd_cmd(s, f'{loc}/design', NUM_DESIGNS - have, start=have), f'{loc}/run_{have}.log')
        print(f'  rc={rc} in {(time.time()-t0)/60:.1f} min')
        if rc: print(tail)
    for f in glob.glob(f'{loc}/design_*'):
        if f.endswith(('.pdb', '.trb')): shutil.copy2(f, drv)
    n = len(glob.glob(f'{drv}/design_*.pdb')); summary.append((name, n)); print(f'  {name}: {n} backbones on Drive')
print('\n', summary)
print('Next: run the Phase 6-8 notebook (ProteinMPNN + Boltz-2 + native comparison).')

  5_1YCR_baseline: 30 backbones on Drive
  5_1YCR_gnn: 30 backbones on Drive
  5_1YCR_oracle: 30 backbones on Drive
  5_1YCR_scrambled: 30 backbones on Drive
  33_1LM8_baseline: 15 backbones on Drive
  33_1LM8_gnn: 15 backbones on Drive
  33_1LM8_oracle: 15 backbones on Drive
  33_1LM8_scrambled: 15 backbones on Drive
  34_2FLU_baseline: 15 backbones on Drive
  34_2FLU_gnn: 15 backbones on Drive
  34_2FLU_oracle: 15 backbones on Drive
  34_2FLU_scrambled: 15 backbones on Drive
  10_2XA0_baseline: 15 backbones on Drive
  10_2XA0_gnn: 15 backbones on Drive

10_2XA0_oracle: 0 done, running 15 more
    [2026-10-16 03:10:51,800][__main__][INFO] - Making design /content/rfdiff_outputs/10_2XA0_oracle/design_0
    [2026-10-16 03:11:44,107][__main__][INFO] - Finished design in 0.87 minutes
    [2026-10-16 03:11:44,107][__main__][INFO] - Making design /content/rfdiff_outputs/10_2XA0_oracle/design_1
    [2026-10-16 03:12:34,734][__main__][INFO] - Finished design in 0.84 minutes
    [2026-10-16 03